In [0]:
from pyspark.sql.functions import *
import pandas as pd

print("=" * 60)
print("REST API HEALTHCARE INGESTION")
print("=" * 60)

# ============================================
# MOCK API RESPONSE
# ============================================

api_data = [

    {
        "patient_id": 101,
        "patient_name": "John Smith",
        "disease": "Diabetes",
        "hospital": "Apollo"
    },

    {
        "patient_id": 102,
        "patient_name": "Sarah Johnson",
        "disease": "Heart Disease",
        "hospital": "Fortis"
    },

    {
        "patient_id": 103,
        "patient_name": "David Miller",
        "disease": "Pneumonia",
        "hospital": "Yashoda"
    },

    {
        "patient_id": 104,
        "patient_name": "Emily Davis",
        "disease": "Hypertension",
        "hospital": "KIMS"
    }

]

# ============================================
# CONVERT API JSON TO DATAFRAME
# ============================================

pdf = pd.DataFrame(api_data)

api_df = spark.createDataFrame(pdf)

print("API Data Loaded Successfully")

display(api_df)

# ============================================
# DATA STANDARDIZATION
# ============================================

standardized_df = api_df.withColumn(
    "load_timestamp",
    current_timestamp()
)

display(standardized_df)

# ============================================
# SAVE TO BRONZE LAYER
# ============================================

standardized_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_migration.bronze.api_healthcare_patients")

print("Bronze API ingestion completed")

# ============================================
# SILVER TRANSFORMATION
# ============================================

silver_df = standardized_df.select(
    upper(col("patient_name")).alias("patient_name"),
    upper(col("disease")).alias("disease"),
    upper(col("hospital")).alias("hospital"),
    "patient_id",
    "load_timestamp"
)

display(silver_df)

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_migration.silver.api_healthcare_patients_cleaned")

print("Silver transformation completed")

# ============================================
# GOLD ANALYTICS
# ============================================

gold_df = silver_df.groupBy("hospital") \
    .agg(count("*").alias("patient_count"))

display(gold_df)

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_migration.gold.api_hospital_summary")

print("Gold analytics completed")

print("=" * 60)
print("REST API INGESTION COMPLETED")
print("=" * 60)

REST API HEALTHCARE INGESTION
API Data Loaded Successfully


patient_id,patient_name,disease,hospital
101,John Smith,Diabetes,Apollo
102,Sarah Johnson,Heart Disease,Fortis
103,David Miller,Pneumonia,Yashoda
104,Emily Davis,Hypertension,KIMS


patient_id,patient_name,disease,hospital,load_timestamp
101,John Smith,Diabetes,Apollo,2026-05-09T07:02:32.480Z
102,Sarah Johnson,Heart Disease,Fortis,2026-05-09T07:02:32.480Z
103,David Miller,Pneumonia,Yashoda,2026-05-09T07:02:32.480Z
104,Emily Davis,Hypertension,KIMS,2026-05-09T07:02:32.480Z


Bronze API ingestion completed


patient_name,disease,hospital,patient_id,load_timestamp
JOHN SMITH,DIABETES,APOLLO,101,2026-05-09T07:02:36.052Z
SARAH JOHNSON,HEART DISEASE,FORTIS,102,2026-05-09T07:02:36.052Z
DAVID MILLER,PNEUMONIA,YASHODA,103,2026-05-09T07:02:36.052Z
EMILY DAVIS,HYPERTENSION,KIMS,104,2026-05-09T07:02:36.052Z


Silver transformation completed


hospital,patient_count
APOLLO,1
FORTIS,1
YASHODA,1
KIMS,1


Gold analytics completed
REST API INGESTION COMPLETED
